In [1]:
import pandas as pd
import numpy as np
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from sklearn.linear_model import LogisticRegression 
from sklearn import svm
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from mlxtend.plotting import plot_sequential_feature_selection as plot_sfs
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Qt5Agg")
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
#Load Dataset
df=pd.read_excel("DATA1800.xlsx")

In [3]:
x = df.drop(columns=['Deaths'])
y = df['Deaths']

In [4]:
metric='roc_auc'

In [ ]:
# Perform Sequential Floating Forward Selection (SFFS) with 5-fold cross-validation using Logistic Regression
sfs1 = SFS(LogisticRegression(),
           k_features=x.shape[1],
           forward=True,
           floating=True,
           scoring=metric,
           cv=5,
           n_jobs=-1)

sfs1.fit(x,y)
df_SFS_results = pd.DataFrame(sfs1.subsets_).transpose()
df_SFS_results.to_excel("SFFS_LR.xlsx",index=False)

lr_score = df_SFS_results.avg_score
fig1 = plot_sfs (sfs1.get_metric_dict(),kind='std_dev')
plt.grid()
plt.show()

In [ ]:
# Perform Sequential Floating Forward Selection (SFFS) with 5-fold cross-validation using Random Forest
sfs2 = SFS(RandomForestClassifier(),
           k_features=x.shape[1],
           forward=True,
           floating=True,
           scoring=metric,
           cv=5,
           n_jobs=-1)
sfs2.fit(x,y)
df_SFS_results=pd.DataFrame(sfs2.subsets_).transpose()
df_SFS_results.to_excel("SFFS_RF.xlsx")
rf_score = df_SFS_results.avg_score 
fig2=plot_sfs(sfs2.get_metric_dict(),kind='std_dev')
plt.grid() 
plt.show()

In [ ]:
# Perform Sequential Floating Forward Selection (SFFS) with 5-fold cross-validation using SVM_linear
sfs3= SFS(svm.SVC(kernel='linear'),
          k_features = x.shape[1],
          forward =True,
          floating =True,
          scoring =metric,
          cv=5)
sfs3.fit(x,y)
df_SFS_results=pd.DataFrame(sfs3.subsets_).transpose()
df_SFS_results.to_excel("SFFS_SVM_linear.xlsx")
svmlr_score = df_SFS_results.avg_score 
fig3=plot_sfs(sfs3.get_metric_dict(),kind='std_dev')
plt.grid() 
plt.show()

In [ ]:
# Perform Sequential Floating Forward Selection (SFFS) with 5-fold cross-validation using SVM_poly
sfs4 = SFS(svm.SVC(kernel='poly'),
           k_features = x.shape[1],
           forward = True,
           floating = True,
           scoring = metric,
           cv = 5)
sfs4.fit(x,y)
df_SFS_results=pd.DataFrame(sfs4.subsets_).transpose()
df_SFS_results.to_excel("SFFS_SVM_poly.xlsx")
svmp_score = df_SFS_results.avg_score 
fig4=plot_sfs(sfs4.get_metric_dict(),kind='std_dev')
plt.grid() 
plt.show()

In [ ]:
# Perform Sequential Floating Forward Selection (SFFS) with 5-fold cross-validation using SVM rbf
sfs5 = SFS(svm.SVC(kernel='rbf'),
           k_features = x.shape[1],
           forward = True,
           floating = True,
           scoring = metric,
           cv = 5)
sfs5.fit(x,y)
df_SFS_results=pd.DataFrame(sfs5.subsets_).transpose()
df_SFS_results.to_excel("SFFS_SVM_rbf.xlsx")
svmr_score = df_SFS_results.avg_score 
fig5=plot_sfs(sfs5.get_metric_dict(),kind='std_dev')
plt.grid() 
plt.show()

In [ ]:
# Perform Sequential Floating Forward Selection (SFFS) with 5-fold cross-validation using MLP
sfs6 = SFS(MLPClassifier(max_iter=2000),
           k_features=x.shape[1],
           forward=True,
           floating=True,
           scoring = metric,
           cv = 5)
#Use SFS to select the top 5 features
sfs6.fit(x, y)

#Create a dataframe for the SFS results
df_SFS_results = pd.DataFrame(sfs6.subsets_).transpose()
df_SFS_results.to_excel("SFFS_MLP.xlsx")
mlp_score = df_SFS_results.avg_score
fig6 = plot_sfs(sfs6.get_metric_dict(), kind='std_dev')

plt.grid()
plt.show()

In [12]:
features = np.arange(24)+1
dic = {'Features': features.tolist(), 'LogR':lr_score,  'RF': rf_score, 'SVM_linear': svmlr_score,
       'SVM_poly': svmp_score, 'SVM_rbf': svmr_score, 'MLP': mlp_score}

df_results = pd.DataFrame(data = dic)
df_results.to_excel("ALL.xlsx")
df_results.plot(x='Features', grid=True, xlabel = ' Number of features', ylabel = 'ROC AUC score', fontsize=14,
                xticks=np.arange(1, 24, 1).tolist(), yticks = np.arange(0.94, 1, 0.02).tolist(), colormap='tab10',
                figsize=(20, 10))
plt.grid()
plt.show()


In [ ]:
import pandas as pd
from scipy.stats import wilcoxon, friedmanchisquare
import matplotlib
matplotlib.use("Qt5Agg")
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
# Wilcoxon signed-rank test
data = pd.read_excel("ALL.xlsx")
data=data.iloc[0:24,1:]
algos =data.columns.tolist()
stat_df = pd.DataFrame(columns=algos,index=algos)
p_df= pd.DataFrame(columns=algos,index=algos)
print(algos)

for algo in algos:
    algo1_data=data[algo].tolist()
    for algo2 in algos:
        if algo2 == algo:
          stat=1
          p_value=1
        else:
          algo2_data=data[algo2].tolist()  
          res= wilcoxon(algo1_data,algo2_data,alternative='greater')
          stat= res.statistic
          p_value= res.pvalue
        stat_df.loc[algo,algo2]= stat
        p_df.loc[algo,algo2]=p_value 

stat_df.to_excel("stat_values.xlsx",index=False)
p_df.to_excel("p_values.xlsx",index=False)    

['LogR', 'RF', 'SVM_linear', 'SVM_poly', 'SVM_rbf', 'MLP']


In [ ]:
# Friedman Chi-Square test
statistic, pvalue = friedmanchisquare(data['LogR'], data['RF'], data['MLP'], data['SVM_poly'], data['SVM_rbf'], data['SVM_linear'])
print(statistic , pvalue)

91.81987577639748 2.7854918549425427e-18


In [ ]:
# Mann-Whitney U test
import pandas as pd
from scipy.stats import mannwhitneyu

x = df.drop(columns=['City' , 'Sex', 'Cough', 'Fever', 'SoreThroat' , 'Aosmia',  'Shiver', 'Notaste', 'Nausea', 'Case_Classification', 'Ventilation','Intubation','Deaths'])
features = x.columns

# List to store the results
results = []

# Compare each feature by category
for feature in features:
    # Split the data into two groups based on class labels
    class_0 = x[feature][y == 0]  
    class_1 = x[feature][y == 1]  
    
    # Perform the Mann-Whitney U test
    U_statistic, p_value = mannwhitneyu(class_0, class_1)
    
   
    results.append((feature, U_statistic, p_value))

for feature, U_statistic, p_value in results:
    print(f"Feature: {feature}, U-statistic: {U_statistic}, p-value: {p_value}")
import pandas as pd

# Create a DataFrame from the test results
df_results = pd.DataFrame(results, columns=['Feature', 'U-statistic', 'p-value'])


df_results.to_excel("mann_whitney_results.xlsx", index=False)

Feature: Age, U-statistic: 48151.5, p-value: 7.757228387412299e-230
Feature: Dates, U-statistic: 540900.0, p-value: 4.3780651595137036e-51
Feature: Myalgia, U-statistic: 487800.0, p-value: 2.8540518521724423e-29
Feature: Diarrhea, U-statistic: 418500.0, p-value: 0.0010449345074944854
Feature: RunningNose, U-statistic: 477900.0, p-value: 4.5562245143428474e-29
Feature: Shortness_Breath, U-statistic: 373050.0, p-value: 3.096715336041267e-08
Feature: Weakness, U-statistic: 467550.0, p-value: 1.752396864894406e-15
Feature: Hospital, U-statistic: 46800.0, p-value: 2.042114359130623e-308
Feature: Morbitity, U-statistic: 325800.0, p-value: 8.323662001169466e-20
Feature: Icu_maf, U-statistic: 292500.0, p-value: 2.0165669249493936e-63
Feature: DOSES , U-statistic: 494100.0, p-value: 3.978609355073578e-21
Feature: Reinfection , U-statistic: 402300.0, p-value: 0.389697433408983


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import scipy.stats as stats
import matplotlib
from sklearn.model_selection import GridSearchCV, cross_val_predict, train_test_split, TimeSeriesSplit, StratifiedKFold
import numpy as np
from sklearn import metrics
matplotlib.use("Qt5Agg")
from sklearn.metrics import classification_report, accuracy_score, f1_score, roc_auc_score, precision_score, \
    recall_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFECV
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

data= pd.read_excel("best_feautures.xlsx")

# RFE features
data_x = data.drop(columns=['Deaths'])
data_y= data['Deaths']


In [ ]:
mlp_model = MLPClassifier(max_iter=1000)
# Define the parameter grid for Grid Search
param_grid={'hidden_layer_sizes': [(5,2), (20,10,10),(10,5), (100, 50, 25)],
    'activation': ['tanh', 'relu'],
    'solver': ['sgd', 'adam'],
    'alpha': [0.0001, 0.05],
    'learning_rate': ['constant','adaptive']}

grid_search = GridSearchCV(estimator=mlp_model, param_grid=param_grid, cv=3, scoring='accuracy', n_jobs=-1)

# Fit Grid Search to the data
grid_search.fit(data_x,data_y)

# Get the best model from grid search
best_model = grid_search.best_estimator_

# 10-Fold Cross-Validation with Predictions
# Use cross_val_predict to get the predictions for the ROC AUC and other metrics
y_pred = cross_val_predict(best_model, data_x, data_y, cv=5)
# Step 6: Compute Metrics
# Accuracy
accuracy = accuracy_score(data_y, y_pred)
print(accuracy)
# F1 Scor
f1 = f1_score(data_y, y_pred)
print(f1)
#ROC-AUC
roc_auc= roc_auc_score(data_y, y_pred)
print(roc_auc)
# precision Score
precision = precision_score(data_y, y_pred)
print(precision)
# recall Score
recall = recall_score(data_y, y_pred)
print(recall)

# Classification Report
report = classification_report(data_y, y_pred,output_dict=True)
df_report = pd.DataFrame(report).transpose()
df_report.to_excel("MLP_report.xlsx")

# save the best parameters
dic = grid_search.best_params_
val = list(dic.values())
keys = list(dic.keys())
best_param = pd.DataFrame(columns=keys)
for key in keys:
    best_param.at[0, key] = dic.get(key)
best_param.to_excel("MLP_bestParams.xlsx", index=False)

0.9577777777777777
0.9577308120133482
0.9577777777777777
0.9587973273942093
0.9566666666666667
